# Easy implementations

readable versions of things the library does in a more efficient way. not used by the library, just kept for understanding.

## basis blades in index notation

each basis blade is written as the list of unit vector indices it is made of, e.g. `[1, 3]` is e13. the library uses bitmasks instead (bit k stands for e_k, so e13 = `0b1010`), see `algebra.py`.

In [1]:
basis_blades_idx = [
    # grade 0 (scalar)
    [],
    # grade 1 (vector)
    [0], [1], [2], [3],
    # grade 2 (bivector)
    [0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3],
    # grade 3 (trivector)
    [0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3],
    # grade 4 (pseudoscalar)
    [0, 1, 2, 3],
]

## basis blade multiplication in index notation

### multiplication index resolver (for 3DPGA only)

concatenating the two index lists is the multiplication. then the indices are bubble sorted, every swap gives a factor (-1), and equal neighbours reduce (e_k e_k = 1, or 0 for e0).

this version only knows the PGA rule (e0² = 0, all others +1). the bitmask version `_resolve_bb_mult_bitmask` does the same with XOR + popcount and works for any metric.

In [2]:
def bb_index_resolve_multiplication(bb1: list[int], bb2: list[int]) -> tuple[int, list[int]]:
    """ resolves the multiplication of two basis blades in index notation. returns (sign, resulting index list). """

    cat = bb1 + bb2 # these are the indices that would result from multiplying two basis blades
    sgn = 0 # this is the exponent of (-1)**sgn, counting the sign changes

    while True:

        chg = False # keeping track if the list has changed this iteration
        rmv = [] # elements to remove from the list at the end of one iteration

        i = 0
        while i < len(cat) - 1:
            a = cat[i]
            b = cat[i + 1]

            # if a == b they reduce
            if a == b:
                if a == 0:
                    # e0 e0 = 0: the entire multiplication collapses to 0 and we're finished
                    return 0, []
                else:
                    # if a = b > 0, their multiplication reduces to 1
                    # skip ahead as this pair is treated at once here. will be removed at the end of the iteration
                    rmv += [i, i + 1]
                    chg = True
                    i += 2
                    continue

            elif a > b:
                # as we defined the basis blades with only increasing indices, we need to swap the two!
                cat[i], cat[i + 1] = cat[i + 1], cat[i]
                # and add a (-1) as all swaps yield a sign change
                chg = True
                sgn += 1
                i += 1
                continue

            # if a < b all good!
            i += 1

        if not chg:
            # if nothing was changed anymore we're done
            return (-1)**sgn, cat

        # remove all rmv elements (basically replace cat)
        cat = [cat[i] for i in range(len(cat)) if i not in rmv]

### check against the bitmask version

both versions have to agree on all 16 × 16 products. if the sign is 0 the resulting blade doesn't matter.

In [3]:
from pga_engine.algebra import PGA3D
from pga_engine.algebra import _bb_bitmask_resolve_multiplication

def idx_to_bitmask(idx: list[int]) -> int:
    return sum(1 << k for k in idx)

for bb1 in basis_blades_idx:
    for bb2 in basis_blades_idx:
        sgn_idx, res_idx = bb_index_resolve_multiplication(bb1, bb2)
        sgn_bit, res_bit = _bb_bitmask_resolve_multiplication(idx_to_bitmask(bb1), idx_to_bitmask(bb2), PGA3D.metric)
        assert sgn_idx == sgn_bit, (bb1, bb2)
        if sgn_idx != 0:
            assert idx_to_bitmask(res_idx) == res_bit, (bb1, bb2)

print(f"all {len(basis_blades_idx) ** 2} products agree")

all 256 products agree


## Looped general geometric product

this version of the geometric product function shows more clearly how it's calculated: a nested loop goes through all the combinatoins of basis blades of the input vectors. so for 2 of them we get a 2d array of their multiplied coefficients (tensor product) But these are only the scalar coefficients... Any multiplied pair of basis blades results in a totally different basis blade - which one can be looked up in the gp_table! 

this 3d dimensional lookup table is a 2d array of one-hot vectors; each being the "result vector" of the multiplication of two basis blades. Additionally, the one-hot vector carries the sign of the result OR is entirely 0 if the multiplication cancels (such as e0^2 under 3DPGA)

In [5]:
import numpy as np
from pga_engine.algebra import Algebra, PGA3D


def gp_loop(v1: np.ndarray, v2: np.ndarray, alg: Algebra = PGA3D) -> np.ndarray:
    """ computes the geometric product for arbitrary multivectors. this is the "more understandable" version that does the calculation through a loop. only works for single multivectors, no batches. used as reference for gp in the tests. """

    v1v2 = np.zeros(alg.dim)

    for i in range(alg.dim):
        for j in range(alg.dim):
            # iterating through each possible multiplication of basis elements. gp_table[i, j] is a whole multivector: zero everywhere except at the resulting blade, where it holds the sign (or all zero if the product vanishes). so it just has to be scaled by both coefficients and added up
            v1v2 += v1[i] * v2[j] * alg.gp_table[i, j]

    return v1v2

### check against einsum geometric product

In [19]:
import numpy as np
from pga_engine.products import gp
from pga_engine.algebra import PGA3D


a = np.random.randn(PGA3D.dim)
b = np.random.randn(PGA3D.dim)

result_gp = gp(a, b, alg=PGA3D)
result_gp_looped = gp_loop(a, b, alg=PGA3D)

print(f"are the gp results the same? -> {np.allclose(result_gp, result_gp_looped)}")

are the gp results the same? -> True
